# AstroLongevity Data Pipeline (Google Colab / Kaggle)

**NASA Space Apps Challenge 2026**

This notebook fetches and processes transcriptomics data from the NASA Open Science Data Repository (OSDR). 
To ensure **strict scientific rigor and zero hallucination**, this pipeline uses explicit validation checks. If data is missing or corrupted, the pipeline will halt immediately.

Target Datasets:
- **OSD-21** (Microarray)
- **OSD-104** (RNA-Seq)
- **OSD-101** (RNA-Seq)

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os

print("Environment initialized. Scientific packages loaded.")

## Step 1: Query NASA OSDR API for Study File Manifests
We will query the official NASA API to find the exact file paths for the processed data (we want normalized count matrices and differential expression tables, avoiding raw FASTQ files).

In [ ]:
def get_study_files(osd_id):
    """Fetches the list of files available for a given study from NASA OSDR."""
    api_url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}"
    print(f"Fetching file manifest for {osd_id}...")
    
    response = requests.get(api_url)
    assert response.status_code == 200, f"CRITICAL ERROR: Failed to fetch files for {osd_id}. HTTP Status: {response.status_code}"
    
    data = response.json()
    file_list = data.get('study_files', [])
    
    assert len(file_list) > 0, f"CRITICAL ERROR: No files found in the payload for {osd_id}."
    return file_list

# Test it on OSD-104
osd104_files = get_study_files("OSD-104")
print(f"SUCCESS: Found {len(osd104_files)} total files for OSD-104.")

## Step 2: Filter and Download Processed Transcriptomic Data
We filter the massive file list to find only the processed CSV/TSV files containing normalized gene counts or differential expression (DE) results.

In [ ]:
def download_processed_data(osd_id, file_list):
    """Filters the file list for processed count/DE data and downloads the first match."""
    target_file = None
    target_url = None
    
    # Look for files that indicate processed RNA-seq or microarray data
    for file_info in file_list:
        fname = file_info.get('file_name', '').lower()
        # We want CSV or TSV files containing normalized counts or differential expression
        if (fname.endswith('.csv') or fname.endswith('.tsv') or fname.endswith('.txt')) and ('rna_seq' in fname or 'microarray' in fname) and ('normalized' in fname or 'differential' in fname):
            target_file = file_info.get('file_name')
            target_url = f"https://osdr.nasa.gov{file_info.get('remote_url')}"
            break
            
    assert target_file is not None, f"CRITICAL ERROR: Could not find processed count matrix for {osd_id}. Manual review required."
    
    print(f"Target identified for {osd_id}: {target_file}")
    print(f"Downloading from {target_url}...")
    
    # Download directly into a pandas DataFrame
    # NASA OSDR files are often comma or tab separated. pandas read_csv handles URLs natively.
    sep = '\t' if target_file.endswith('.tsv') or target_file.endswith('.txt') else ','
    try:
        df = pd.read_csv(target_url, sep=sep)
        return df, target_file
    except Exception as e:
        raise RuntimeError(f"CRITICAL ERROR: Failed to parse {target_file} into a DataFrame. Error: {e}")

# We will test downloading the processed data for OSD-104
# NOTE: In Colab, you can uncomment the lines below to execute the download.
# df_104, filename_104 = download_processed_data("OSD-104", osd104_files)
# print(f"SUCCESS: Downloaded {filename_104} with shape {df_104.shape}")

## Step 3: Strict Validation & QC Gates
Before proceeding with any signature reversal, we must mathematically prove the data is intact. No missing values allowed in critical columns, and the gene list must be robust.

In [ ]:
def validate_dataframe(df, dataset_name):
    """Runs mathematical validation gates on the DataFrame."""
    print(f"\n--- Running QC Validation for {dataset_name} ---")
    
    # Gate 1: Check for empty dataframe
    row_count, col_count = df.shape
    assert row_count > 1000, f"FAIL: Dataset {dataset_name} has only {row_count} rows. Expected >1000 genes."
    assert col_count > 2, f"FAIL: Dataset {dataset_name} has only {col_count} columns. Missing sample data."
    print(f"PASS: Matrix dimensions valid ({row_count} genes, {col_count} columns).")
    
    # Gate 2: Check for missing values in the index/genes
    # Assuming the first column contains the gene symbols or Ensembl IDs
    first_col = df.columns[0]
    missing_genes = df[first_col].isna().sum()
    assert missing_genes == 0, f"FAIL: Dataset {dataset_name} contains {missing_genes} missing gene identifiers."
    print(f"PASS: No missing gene identifiers detected.")
    
    print("STATUS: Data passed strict scientific validation. Ready for downstream pipeline.")

# In Colab, uncomment to run validation:
# validate_dataframe(df_104, "OSD-104")